In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 81.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 76.4 MB/s eta 0:00:00:00:01


In [3]:
!pip install -q datasets peft -U
from datasets import load_dataset

ds = load_dataset("MM-Hallu/POPE")
print(ds)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 10.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 27.4 MB/s eta 0:00:00


README.md:   0%|          | 0.00/524 [00:00<?, ?B/s]

data/test-00000-of-00003.parquet:   0%|          | 0.00/85.0M [00:00<?, ?B/s]

data/test-00001-of-00003.parquet:   0%|          | 0.00/85.0M [00:00<?, ?B/s]

data/test-00002-of-00003.parquet:   0%|          | 0.00/85.0M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['id', 'question_id', 'question', 'answer', 'image_source', 'image', 'category'],
        num_rows: 9000
    })
})


In [4]:
import numpy as np
import pandas as pd

pope = ds["test"]

meta = pope.select_columns(["id", "question_id", "question", "answer", "image_source", "category"]).to_pandas()
meta["label"] = (meta["answer"].str.lower().str.strip() == "yes").astype(int)
meta["row_idx"] = np.arange(len(meta))

print(meta.shape)
print(meta["category"].value_counts())

(9000, 8)
category
adversarial    3000
popular        3000
random         3000
Name: count, dtype: int64


In [5]:
import os
import gc
import torch

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

# Defined once, early -- every later cell can rely on these existing,
# regardless of run order.
OUT_DIR = "/kaggle/working"
BASELINE_PATH = f"{OUT_DIR}/baseline_subset.parquet"
os.makedirs(OUT_DIR, exist_ok=True)


cuda Tesla T4


In [6]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

def load_fresh_model(adapter_path=None):
    for varname in ["model", "train_model", "lora_model", "ablation_model", "plain_model"]:
        if varname in globals():
            del globals()[varname]
    gc.collect()
    torch.cuda.empty_cache()

    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=64*28*28, max_pixels=128*28*28)
    m = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
    ).to("cuda:0")

    if adapter_path is not None:
        from peft import PeftModel
        m = PeftModel.from_pretrained(m, adapter_path)

    return m, processor

model, processor = load_fresh_model()
tok = processor.tokenizer
print("Loaded Qwen2-VL-2B. Device:", next(model.parameters()).device)


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Loaded Qwen2-VL-2B. Device: cuda:0


In [7]:
def get_variant_ids(words):
    ids = set()
    for w in words:
        for variant in (w, " " + w, w.lower(), " " + w.lower()):
            enc = tok.encode(variant, add_special_tokens=False)
            if len(enc) >= 1:
                ids.add(enc[-1])
    return sorted(ids)

YES_IDS = get_variant_ids(["Yes"])
NO_IDS = get_variant_ids(["No"])
print("YES token ids:", YES_IDS, [tok.decode([i]) for i in YES_IDS])
print("NO  token ids:", NO_IDS, [tok.decode([i]) for i in NO_IDS])

YES token ids: [7414, 9454, 9693, 9834] [' Yes', 'Yes', 'yes', ' yes']
NO  token ids: [902, 2152, 2308, 2753] [' no', 'no', ' No', 'No']


In [8]:
def build_prompt(question: str) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"{question} Answer the question using a single word, yes or no."},
        ],
    }]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [9]:
from tqdm.auto import tqdm

@torch.inference_mode()
def score_batch(images, questions, model, batch_size=16):
    all_logit_yes, all_logit_no = [], []
    total_batches = (len(questions) + batch_size - 1) // batch_size

    for i in tqdm(range(0, len(questions), batch_size), total=total_batches, desc="Scoring"):
        batch_imgs = images[i:i+batch_size]
        batch_qs = questions[i:i+batch_size]
        prompts = [build_prompt(q) for q in batch_qs]

        inputs = processor(
            images=batch_imgs, text=prompts, return_tensors="pt", padding=True,
        ).to(model.device, torch.bfloat16)

        out = model(**inputs)
        attn = inputs["attention_mask"]
        last_idx = attn.sum(dim=1) - 1
        logits = out.logits[torch.arange(len(batch_qs)), last_idx, :].float()

        all_logit_yes.append(logits[:, YES_IDS].max(dim=-1).values.cpu())
        all_logit_no.append(logits[:, NO_IDS].max(dim=-1).values.cpu())

    logit_yes = torch.cat(all_logit_yes).numpy()
    logit_no = torch.cat(all_logit_no).numpy()
    prob_yes = 1 / (1 + np.exp(-(logit_yes - logit_no)))
    pred = (logit_yes > logit_no).astype(int)

    return pd.DataFrame({"logit_yes": logit_yes, "logit_no": logit_no, "prob_yes": prob_yes, "pred": pred})

In [10]:
def compute_metrics(df, label_col="label", pred_col="pred", group_col=None):
    def _row(g):
        tp = ((g[pred_col] == 1) & (g[label_col] == 1)).sum()
        fp = ((g[pred_col] == 1) & (g[label_col] == 0)).sum()
        tn = ((g[pred_col] == 0) & (g[label_col] == 0)).sum()
        fn = ((g[pred_col] == 0) & (g[label_col] == 1)).sum()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)
        return dict(n=len(g), accuracy=(tp+tn)/len(g), fpr=fp/max(fp+tn, 1), yes_rate=g[pred_col].mean(),
                    precision=precision, recall=recall, f1=f1, tp=tp, fp=fp, tn=tn, fn=fn)

    if group_col is None:
        return pd.DataFrame([_row(df)])
    rows = [{**{"split": name}, **_row(g)} for name, g in df.groupby(group_col)]
    rows.append({"split": "ALL", **_row(df)})
    return pd.DataFrame(rows).round(4)

In [11]:
sample = pope.select(range(5))
sample_df = score_batch(sample["image"], sample["question"], model)
sample_df["label"] = [1 if a.lower().strip() == "yes" else 0 for a in sample["answer"]]
print(sample_df)
print(compute_metrics(sample_df))

Scoring:   0%|          | 0/1 [00:00<?, ?it/s]

   logit_yes  logit_no  prob_yes  pred  label
0     23.500   22.0000  0.817575     1      1
1     15.750   16.0000  0.437823     0      0
2     17.125   14.5625  0.928409     1      1
3     16.125   16.7500  0.348645     0      0
4     23.000   22.1250  0.705785     1      1
   n  accuracy  fpr  yes_rate  precision  recall   f1  tp  fp  tn  fn
0  5       1.0  0.0       0.6        1.0     1.0  1.0   3   0   2   0


In [12]:
# ============================================================
# PART 2 -- Grouped, stratified split (train / dev / test), by
# image_source so no image appears in more than one split.
# Each split is then capped to a fixed per-category sample size,
# which is the change that keeps every later cell fast.
# ============================================================
TRAIN_PER_CATEGORY = 100_000   # -> 900 training rows
DEV_PER_CATEGORY = 100_000     # -> 300 dev rows
TEST_PER_CATEGORY = 100_000    # -> 300 test rows

def make_splits(meta_df, train_frac=0.70, dev_frac=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    img_cat = meta_df.groupby("image_source")["category"].agg(lambda x: x.mode()[0])
    splits = {}
    for cat, group in img_cat.groupby(img_cat):
        imgs = group.index.values.copy()
        rng.shuffle(imgs)
        n = len(imgs)
        n_train = int(n * train_frac)
        n_dev = int(n * dev_frac)
        for img in imgs[:n_train]:
            splits[img] = "train"
        for img in imgs[n_train:n_train + n_dev]:
            splits[img] = "dev"
        for img in imgs[n_train + n_dev:]:
            splits[img] = "test"
    return splits

img_splits = make_splits(meta)
meta["split"] = meta["image_source"].map(img_splits)

train_imgs = set(meta.loc[meta.split == "train", "image_source"])
dev_imgs = set(meta.loc[meta.split == "dev", "image_source"])
test_imgs = set(meta.loc[meta.split == "test", "image_source"])
assert not (train_imgs & dev_imgs) and not (train_imgs & test_imgs) and not (dev_imgs & test_imgs), "LEAK across splits"
print("Split sizes (rows):", meta["split"].value_counts().to_dict())
print("No leakage across splits: confirmed.")

def subsample(meta_df, split_name, per_category, seed=SEED):
    sub = meta_df[meta_df["split"] == split_name]
    return (
        sub.groupby("category", group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), per_category), random_state=seed))
        .reset_index(drop=True)
    )

train_meta = subsample(meta, "train", TRAIN_PER_CATEGORY)
dev_meta = subsample(meta, "dev", DEV_PER_CATEGORY)
test_meta = subsample(meta, "test", TEST_PER_CATEGORY)

train_rows = pope.select(train_meta["row_idx"].tolist())
dev_rows = pope.select(dev_meta["row_idx"].tolist())
test_rows = pope.select(test_meta["row_idx"].tolist())

print("Using: train", len(train_meta), "| dev", len(dev_meta), "| test", len(test_meta))
meta.to_parquet(f"{OUT_DIR}/pope_meta_with_splits.parquet")

Split sizes (rows): {'train': 6300, 'test': 1350, 'dev': 1350}
No leakage across splits: confirmed.
Using: train 6300 | dev 1350 | test 1350


/tmp/ipykernel_107/1200383817.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), per_category), random_state=seed))
/tmp/ipykernel_107/1200383817.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), per_category), random_state=seed))
/tmp/ipykernel_107/1200383817.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping colu

In [13]:
# ============================================================
# PART 3 -- Baseline eval, BEFORE any fine-tuning.
# Scored only on dev+test (600 rows) -- exactly what Part 7 needs
# for a fair before/after comparison, and enough for the Part 4
# diagnostic analysis. Cached to disk so re-running this cell
# after any later crash/restart is instant, not another full run.
# ============================================================
if os.path.exists(BASELINE_PATH):
    baseline_df = pd.read_parquet(BASELINE_PATH)
    print("Loaded cached baseline from disk. Shape:", baseline_df.shape)
else:
    eval_meta = meta.reset_index(drop=True)   # full POPE dataset, all 9000 rows
    eval_rows = pope

    baseline_df = score_batch(eval_rows["image"], eval_rows["question"], model, batch_size=16)
    baseline_df["label"] = eval_meta["label"].values
    baseline_df["category"] = eval_meta["category"].values
    baseline_df["image_source"] = eval_meta["image_source"].values
    baseline_df["question"] = eval_meta["question"].values
    baseline_df["split"] = eval_meta["split"].values
    baseline_df.to_parquet(BASELINE_PATH)
    print("Ran baseline inference on dev+test (600 rows), cached to disk.")

print(baseline_df.shape)
baseline_metrics = compute_metrics(baseline_df, group_col="category")
print(baseline_metrics)
baseline_metrics.to_csv(f"{OUT_DIR}/baseline_metrics.csv", index=False)

Scoring:   0%|          | 0/563 [00:00<?, ?it/s]

Ran baseline inference on dev+test (600 rows), cached to disk.
(9000, 9)
         split     n  accuracy     fpr  yes_rate  precision  recall      f1  \
0  adversarial  3000    0.7653  0.2760    0.5413     0.7451  0.8067  0.7746   
1      popular  3000    0.8030  0.2133    0.5163     0.7934  0.8193  0.8062   
2       random  3000    0.7857  0.2707    0.5563     0.7567  0.8420  0.7971   
3          ALL  9000    0.7847  0.2533    0.5380     0.7646  0.8227  0.7925   

     tp    fp    tn   fn  
0  1210   414  1086  290  
1  1229   320  1180  271  
2  1263   406  1094  237  
3  3702  1140  3360  798  


In [14]:
# ============================================================
# PART 4 -- Diagnostic failure analysis (pure pandas, no GPU).
# ============================================================
import re

fp_mask = (baseline_df["pred"] == 1) & (baseline_df["label"] == 0)
tp_mask = (baseline_df["pred"] == 1) & (baseline_df["label"] == 1)
fn_mask = (baseline_df["pred"] == 0) & (baseline_df["label"] == 1)
tn_mask = (baseline_df["pred"] == 0) & (baseline_df["label"] == 0)

print("=== Confidence: hallucinated vs correct 'yes' ===")
print(f"Mean prob_yes | False Positive: {baseline_df.loc[fp_mask, 'prob_yes'].mean():.4f} (n={fp_mask.sum()})")
print(f"Mean prob_yes | True Positive:  {baseline_df.loc[tp_mask, 'prob_yes'].mean():.4f} (n={tp_mask.sum()})")
print(f"Mean prob_yes | False Negative: {baseline_df.loc[fn_mask, 'prob_yes'].mean():.4f} (n={fn_mask.sum()})")
print(f"Mean prob_yes | True Negative:  {baseline_df.loc[tn_mask, 'prob_yes'].mean():.4f} (n={tn_mask.sum()})")

def extract_object(question):
    m = re.search(r"a[n]? ([\w\s]+?) in the image", question, flags=re.IGNORECASE)
    return m.group(1).strip().lower() if m else None

baseline_df["object"] = baseline_df["question"].apply(extract_object)
fp_objects = baseline_df.loc[fp_mask, "object"].value_counts()
all_objects = baseline_df["object"].value_counts()

obj_stats = pd.DataFrame({"n_asked": all_objects, "n_false_positive": fp_objects}).fillna(0)
obj_stats["fp_rate_of_asks"] = (obj_stats["n_false_positive"] / obj_stats["n_asked"]).round(3)
obj_stats = obj_stats[obj_stats["n_asked"] >= 3]
print("\n=== Most-hallucinated objects ===")
print(obj_stats.sort_values("fp_rate_of_asks", ascending=False).head(15))

obj_stats.sort_values("fp_rate_of_asks", ascending=False).to_csv(f"{OUT_DIR}/diagnosis_object_fpr.csv")
baseline_df.to_parquet(f"{OUT_DIR}/baseline_with_diagnosis.parquet")
print("\nSaved diagnosis outputs to", OUT_DIR)

=== Confidence: hallucinated vs correct 'yes' ===
Mean prob_yes | False Positive: 0.7734 (n=1140)
Mean prob_yes | True Positive:  0.8599 (n=3702)
Mean prob_yes | False Negative: 0.2906 (n=798)
Mean prob_yes | True Negative:  0.2409 (n=3360)

=== Most-hallucinated objects ===
           n_asked  n_false_positive  fp_rate_of_asks
object                                               
scissors        29              10.0            0.345
tie             40              13.0            0.325
keyboard        27               8.0            0.296
cow             34              10.0            0.294
bear            38              11.0            0.289
elephant        21               6.0            0.286
suitcase        40              11.0            0.275
orange          51              14.0            0.275
sheep           22               6.0            0.273
chair          591             145.0            0.245
carrot          37               9.0            0.243
bird            58    

In [15]:
# ============================================================
# PART 5 -- LoRA training data (train split only; dev/test never touched here)
# ============================================================
print("Category balance in training data:")
print(train_meta["category"].value_counts())
print("\nLabel balance:")
print(train_meta["label"].value_counts())

for i in range(2):
    q, a = train_meta.loc[i, "question"], train_meta.loc[i, "answer"]
    p = build_prompt(q)
    t = " " + a.strip().capitalize()
    print(f"PROMPT: {p}\nTARGET: '{t}'\n")

train_meta.to_parquet(f"{OUT_DIR}/train_meta.parquet")

Category balance in training data:
category
adversarial    2100
popular        2100
random         2100
Name: count, dtype: int64

Label balance:
label
1    3150
0    3150
Name: count, dtype: int64
PROMPT: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Is there a bed in the image? Answer the question using a single word, yes or no.<|im_end|>
<|im_start|>assistant

TARGET: ' Yes'

PROMPT: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Is there a person in the image? Answer the question using a single word, yes or no.<|im_end|>
<|im_start|>assistant

TARGET: ' Yes'



In [16]:
# ============================================================
# PART 6 -- LoRA training
# ============================================================
del model
gc.collect()
torch.cuda.empty_cache()

train_model, train_processor = load_fresh_model()
train_tok = train_processor.tokenizer
print("Loaded a fresh trainable copy of the base model.")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loaded a fresh trainable copy of the base model.


In [17]:
!pip uninstall -y -q torchao

In [18]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

train_model = get_peft_model(train_model, lora_config)
train_model.print_trainable_parameters()

trainable params: 17,432,576 || all params: 2,226,418,176 || trainable%: 0.7830


In [19]:
from torch.utils.data import Dataset, DataLoader

class PopeTrainingDataset(Dataset):
    def __init__(self, hf_rows, processor):
        self.rows = hf_rows
        self.processor = processor

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        question, answer = row["question"], row["answer"]

        prompt_text = build_prompt(question)
        target_text = " " + answer.strip().capitalize()
        full_text = prompt_text + target_text

        encoded = self.processor(images=row["image"], text=full_text, return_tensors="pt")

        input_ids = encoded["input_ids"][0]
        pixel_values = encoded["pixel_values"]
        image_grid_thw = encoded["image_grid_thw"][0]
        mm_token_type_ids = encoded["mm_token_type_ids"][0]

        prompt_only_ids = self.processor.tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        num_prompt_tokens = len(prompt_only_ids)

        labels = input_ids.clone()
        labels[:num_prompt_tokens] = -100

        return {
            "input_ids": input_ids,
            "pixel_values": pixel_values,
            "image_grid_thw": image_grid_thw,
            "mm_token_type_ids": mm_token_type_ids,
            "labels": labels,
        }

In [20]:
def collate_batch(batch_of_examples):
    max_length = max(len(example["input_ids"]) for example in batch_of_examples)
    pad_token_id = train_tok.pad_token_id or train_tok.eos_token_id

    padded_input_ids, padded_labels, padded_mm_token_type_ids, attention_masks = [], [], [], []

    for example in batch_of_examples:
        current_length = len(example["input_ids"])
        pad_amount = max_length - current_length
        padded_input_ids.append(torch.cat([example["input_ids"], torch.full((pad_amount,), pad_token_id, dtype=torch.long)]))
        padded_labels.append(torch.cat([example["labels"], torch.full((pad_amount,), -100, dtype=torch.long)]))
        padded_mm_token_type_ids.append(torch.cat([example["mm_token_type_ids"], torch.zeros(pad_amount, dtype=torch.long)]))
        attention_masks.append(torch.cat([torch.ones(current_length, dtype=torch.long), torch.zeros(pad_amount, dtype=torch.long)]))

    pixel_values = torch.cat([example["pixel_values"] for example in batch_of_examples], dim=0)
    image_grid_thw = torch.stack([example["image_grid_thw"] for example in batch_of_examples])

    return {
        "input_ids": torch.stack(padded_input_ids),
        "attention_mask": torch.stack(attention_masks),
        "labels": torch.stack(padded_labels),
        "pixel_values": pixel_values,
        "image_grid_thw": image_grid_thw,
        "mm_token_type_ids": torch.stack(padded_mm_token_type_ids),
    }

train_dataset = PopeTrainingDataset(train_rows, train_processor)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_batch)
print("Training batches per epoch:", len(train_loader))


Training batches per epoch: 1575


In [21]:
CHECKPOINT_DIR = f"{OUT_DIR}/lora_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_EVERY = 100

optimizer = torch.optim.AdamW(train_model.parameters(), lr=2e-4)
NUM_EPOCHS = 1

train_model.train()
step_number = 0

for epoch in range(NUM_EPOCHS):
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        batch = {k: v.to(train_model.device) for k, v in batch.items()}
        batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)

        outputs = train_model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        step_number += 1
        progress_bar.set_postfix(loss=loss.item())

        if step_number % CHECKPOINT_EVERY == 0:
            train_model.save_pretrained(f"{CHECKPOINT_DIR}/step_{step_number}")

print("Training complete.")
ADAPTER_PATH = f"{OUT_DIR}/lora_pope_adapter"
train_model.save_pretrained(ADAPTER_PATH)
print("Final adapter saved to:", ADAPTER_PATH)


Epoch 1:   0%|          | 0/1575 [00:00<?, ?it/s]

Training complete.
Final adapter saved to: /kaggle/working/lora_pope_adapter


In [22]:
# ============================================================
# PART 7 -- Evaluate the LoRA-adapted model
# ============================================================
del train_model
gc.collect()
torch.cuda.empty_cache()

lora_model, lora_processor = load_fresh_model(adapter_path=ADAPTER_PATH)
lora_tok = lora_processor.tokenizer
print("Loaded base model + LoRA adapter.")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loaded base model + LoRA adapter.


In [23]:
print("Evaluating LoRA model on dev split:", len(dev_meta), "examples")
dev_lora_df = score_batch(dev_rows["image"], dev_rows["question"], lora_model, batch_size=16)
dev_lora_df["label"] = dev_meta["label"].values
dev_lora_df["category"] = dev_meta["category"].values
dev_lora_df.to_parquet(f"{OUT_DIR}/dev_lora_predictions.parquet")

dev_baseline_df = baseline_df[baseline_df["split"] == "dev"].reset_index(drop=True)

print("\n=== BASELINE on dev ===")
print(compute_metrics(dev_baseline_df, group_col="category"))
print("\n=== LoRA on dev ===")
print(compute_metrics(dev_lora_df, group_col="category"))


Evaluating LoRA model on dev split: 1350 examples


Scoring:   0%|          | 0/85 [00:00<?, ?it/s]


=== BASELINE on dev ===
         split     n  accuracy     fpr  yes_rate  precision  recall      f1  \
0  adversarial   450    0.7689  0.2578    0.5267     0.7553  0.7956  0.7749   
1      popular   450    0.7911  0.2533    0.5444     0.7673  0.8356  0.8000   
2       random   450    0.7822  0.2889    0.5711     0.7471  0.8533  0.7967   
3          ALL  1350    0.7807  0.2667    0.5474     0.7564  0.8281  0.7907   

    tp   fp   tn   fn  
0  179   58  167   46  
1  188   57  168   37  
2  192   65  160   33  
3  559  180  495  116  

=== LoRA on dev ===
         split     n  accuracy     fpr  yes_rate  precision  recall      f1  \
0  adversarial   450    0.5489  0.5867    0.6356     0.5385  0.6844  0.6027   
1      popular   450    0.5667  0.5911    0.6578     0.5507  0.7244  0.6257   
2       random   450    0.6000  0.4400    0.5400     0.5926  0.6400  0.6154   
3          ALL  1350    0.5719  0.5393    0.6111     0.5588  0.6830  0.6147   

    tp   fp   tn   fn  
0  154  132   93  

In [24]:
print("Evaluating LoRA model on TEST split (final numbers):", len(test_meta), "examples")
test_lora_df = score_batch(test_rows["image"], test_rows["question"], lora_model, batch_size=16)
test_lora_df["label"] = test_meta["label"].values
test_lora_df["category"] = test_meta["category"].values
test_lora_df.to_parquet(f"{OUT_DIR}/test_lora_predictions.parquet")

test_baseline_df = baseline_df[baseline_df["split"] == "test"].reset_index(drop=True)

print("\n=== FINAL: BASELINE on test ===")
print(compute_metrics(test_baseline_df, group_col="category"))
print("\n=== FINAL: LoRA on test ===")
print(compute_metrics(test_lora_df, group_col="category"))

Evaluating LoRA model on TEST split (final numbers): 1350 examples


Scoring:   0%|          | 0/85 [00:00<?, ?it/s]


=== FINAL: BASELINE on test ===
         split     n  accuracy     fpr  yes_rate  precision  recall      f1  \
0  adversarial   450    0.7622  0.2444    0.5067     0.7588  0.7689  0.7638   
1      popular   450    0.8022  0.2000    0.5022     0.8009  0.8044  0.8027   
2       random   450    0.7667  0.2756    0.5422     0.7459  0.8089  0.7761   
3          ALL  1350    0.7770  0.2400    0.5170     0.7679  0.7941  0.7808   

    tp   fp   tn   fn  
0  173   55  170   52  
1  181   45  180   44  
2  182   62  163   43  
3  536  162  513  139  

=== FINAL: LoRA on test ===
         split     n  accuracy     fpr  yes_rate  precision  recall      f1  \
0  adversarial   450    0.5378  0.6178    0.6556     0.5288  0.6933  0.6000   
1      popular   450    0.5667  0.5822    0.6489     0.5514  0.7156  0.6228   
2       random   450    0.5889  0.5111    0.6000     0.5741  0.6889  0.6263   
3          ALL  1350    0.5644  0.5704    0.6348     0.5508  0.6993  0.6162   

    tp   fp   tn   fn  
0 

In [25]:
# ============================================================
# ABLATION -- does prompting alone ("be careful") work as well as
# real LoRA training? Uses the plain baseline model + a stricter
# prompt, run on the same test set.
# ============================================================
def cautious_prompt(question: str) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"{question} Be very careful and only answer yes if you are certain the object is visible. Answer yes or no."},
        ],
    }]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def score_batch_with_custom_prompt(images, questions, model, processor_for_this, prompt_fn, yes_ids, no_ids, batch_size=16):
    all_logit_yes, all_logit_no = [], []
    for i in tqdm(range(0, len(questions), batch_size), desc="Scoring (cautious prompt)"):
        batch_images = images[i:i+batch_size]
        batch_questions = questions[i:i+batch_size]
        prompts = [prompt_fn(q) for q in batch_questions]
        inputs = processor_for_this(images=batch_images, text=prompts, return_tensors="pt", padding=True).to(model.device, torch.bfloat16)
        outputs = model(**inputs)
        attention_mask = inputs["attention_mask"]
        last_token_position = attention_mask.sum(dim=1) - 1
        logits = outputs.logits[torch.arange(len(batch_questions)), last_token_position, :].float()
        all_logit_yes.append(logits[:, yes_ids].max(dim=-1).values.cpu())
        all_logit_no.append(logits[:, no_ids].max(dim=-1).values.cpu())
    logit_yes = torch.cat(all_logit_yes).numpy()
    logit_no = torch.cat(all_logit_no).numpy()
    prob_yes = 1 / (1 + np.exp(-(logit_yes - logit_no)))
    pred = (logit_yes > logit_no).astype(int)
    return pd.DataFrame({"logit_yes": logit_yes, "logit_no": logit_no, "prob_yes": prob_yes, "pred": pred})

del lora_model
gc.collect()
torch.cuda.empty_cache()
ablation_model, ablation_processor = load_fresh_model()

prompt_only_df = score_batch_with_custom_prompt(
    test_rows["image"], test_rows["question"], ablation_model, ablation_processor,
    cautious_prompt, YES_IDS, NO_IDS, batch_size=16,
)
prompt_only_df["label"] = test_meta["label"].values
prompt_only_df["category"] = test_meta["category"].values

print("\n=== ABLATION: prompt-only 'be careful' on test ===")
print(compute_metrics(prompt_only_df, group_col="category"))

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Scoring (cautious prompt):   0%|          | 0/85 [00:00<?, ?it/s]


=== ABLATION: prompt-only 'be careful' on test ===
         split     n  accuracy     fpr  yes_rate  precision  recall      f1  \
0  adversarial   450    0.6533  0.3644    0.5178     0.6481  0.6711  0.6594   
1      popular   450    0.6644  0.3600    0.5244     0.6568  0.6889  0.6725   
2       random   450    0.6956  0.2933    0.4889     0.7000  0.6844  0.6921   
3          ALL  1350    0.6711  0.3393    0.5104     0.6676  0.6815  0.6745   

    tp   fp   tn   fn  
0  151   82  143   74  
1  155   81  144   70  
2  154   66  159   71  
3  460  229  446  215  


In [26]:
summary_table = pd.concat([
    compute_metrics(test_baseline_df, group_col="category").assign(method="baseline"),
    compute_metrics(prompt_only_df, group_col="category").assign(method="prompt_only"),
    compute_metrics(test_lora_df, group_col="category").assign(method="lora"),
])
summary_table.to_csv(f"{OUT_DIR}/final_summary_table.csv", index=False)
print(summary_table)


         split     n  accuracy     fpr  yes_rate  precision  recall      f1  \
0  adversarial   450    0.7622  0.2444    0.5067     0.7588  0.7689  0.7638   
1      popular   450    0.8022  0.2000    0.5022     0.8009  0.8044  0.8027   
2       random   450    0.7667  0.2756    0.5422     0.7459  0.8089  0.7761   
3          ALL  1350    0.7770  0.2400    0.5170     0.7679  0.7941  0.7808   
0  adversarial   450    0.6533  0.3644    0.5178     0.6481  0.6711  0.6594   
1      popular   450    0.6644  0.3600    0.5244     0.6568  0.6889  0.6725   
2       random   450    0.6956  0.2933    0.4889     0.7000  0.6844  0.6921   
3          ALL  1350    0.6711  0.3393    0.5104     0.6676  0.6815  0.6745   
0  adversarial   450    0.5378  0.6178    0.6556     0.5288  0.6933  0.6000   
1      popular   450    0.5667  0.5822    0.6489     0.5514  0.7156  0.6228   
2       random   450    0.5889  0.5111    0.6000     0.5741  0.6889  0.6263   
3          ALL  1350    0.5644  0.5704    0.6348    

In [27]:
# ============================================================
# PART 8 -- A-OKVQA capability retention check (optional; skip
# this whole section if you are short on time -- it is a stretch
# goal, not core to the PS's primary/secondary objectives).
# ============================================================
from datasets import load_dataset as load_dataset_aokvqa

aokvqa = load_dataset_aokvqa("HuggingFaceM4/A-OKVQA", split="validation")
NUM_AOKVQA_EXAMPLES = 100
aokvqa_sample = aokvqa.select(range(min(NUM_AOKVQA_EXAMPLES, len(aokvqa))))
aokvqa_examples = [aokvqa_sample[i] for i in range(len(aokvqa_sample))]
print("A-OKVQA sample size:", len(aokvqa_examples))
print("Example row keys:", aokvqa_examples[0].keys())


README.md:   0%|          | 0.00/964 [00:00<?, ?B/s]

data/train-00000-of-00002-c1d24de3bacb5e(…):   0%|          | 0.00/462M [00:00<?, ?B/s]

data/train-00001-of-00002-6b4f3abe2dc385(…):   0%|          | 0.00/464M [00:00<?, ?B/s]

data/validation-00000-of-00001-b2bd0de23(…):   0%|          | 0.00/60.4M [00:00<?, ?B/s]

data/test-00000-of-00001-d306bf3ad53b661(…):   0%|          | 0.00/337M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17056 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1145 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6702 [00:00<?, ? examples/s]

A-OKVQA sample size: 100
Example row keys: dict_keys(['image', 'question_id', 'question', 'choices', 'correct_choice_idx', 'direct_answers', 'difficult_direct_answer', 'rationales'])


In [28]:
def build_aokvqa_prompt(question: str, choices: list) -> str:
    choice_labels = ["A", "B", "C", "D"]
    choice_text = " ".join(f"({label}) {choice}" for label, choice in zip(choice_labels, choices))
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"{question} Choose one: {choice_text} Answer with the letter only."},
        ],
    }]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def evaluate_aokvqa(model, processor_for_this, examples, batch_size=4):
    num_correct, num_total = 0, 0
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        images = [item["image"] for item in batch]
        prompts = [build_aokvqa_prompt(item["question"], item["choices"]) for item in batch]

        inputs = processor_for_this(images=images, text=prompts, return_tensors="pt", padding=True).to(model.device, torch.bfloat16)
        generated = model.generate(**inputs, max_new_tokens=3, do_sample=False)

        prompt_length = inputs["input_ids"].shape[1]
        new_tokens = generated[:, prompt_length:]
        decoded_answers = processor_for_this.batch_decode(new_tokens, skip_special_tokens=True)

        for item, predicted_text in zip(batch, decoded_answers):
            predicted_letter = predicted_text.strip()[:1].upper()
            correct_letter = ["A", "B", "C", "D"][item["correct_choice_idx"]]
            if predicted_letter == correct_letter:
                num_correct += 1
            num_total += 1
    return num_correct / num_total

In [29]:
del ablation_model
gc.collect()
torch.cuda.empty_cache()

plain_model, plain_processor = load_fresh_model()
print("Evaluating BASELINE model on A-OKVQA...")
baseline_aokvqa_accuracy = evaluate_aokvqa(plain_model, plain_processor, aokvqa_examples)
print("Baseline A-OKVQA accuracy:", baseline_aokvqa_accuracy)

del plain_model
gc.collect()
torch.cuda.empty_cache()

lora_model_for_aokvqa, lora_processor_for_aokvqa = load_fresh_model(adapter_path=ADAPTER_PATH)
print("Evaluating LoRA model on A-OKVQA...")
lora_aokvqa_accuracy = evaluate_aokvqa(lora_model_for_aokvqa, lora_processor_for_aokvqa, aokvqa_examples)
print("LoRA A-OKVQA accuracy:", lora_aokvqa_accuracy)

print("\n=== A-OKVQA capability retention ===")
print(f"Baseline accuracy: {baseline_aokvqa_accuracy:.3f}")
print(f"LoRA accuracy:     {lora_aokvqa_accuracy:.3f}")
print(f"Difference:        {lora_aokvqa_accuracy - baseline_aokvqa_accuracy:+.3f}")

capability_summary = pd.DataFrame([{
    "task": "A-OKVQA (multiple choice VQA)",
    "baseline_accuracy": baseline_aokvqa_accuracy,
    "lora_accuracy": lora_aokvqa_accuracy,
    "difference": lora_aokvqa_accuracy - baseline_aokvqa_accuracy,
}])
capability_summary.to_csv(f"{OUT_DIR}/aokvqa_capability_check.csv", index=False)
print(capability_summary)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Evaluating BASELINE model on A-OKVQA...
Baseline A-OKVQA accuracy: 0.79


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Evaluating LoRA model on A-OKVQA...
LoRA A-OKVQA accuracy: 0.4

=== A-OKVQA capability retention ===
Baseline accuracy: 0.790
LoRA accuracy:     0.400
Difference:        -0.390
                            task  baseline_accuracy  lora_accuracy  difference
0  A-OKVQA (multiple choice VQA)               0.79            0.4       -0.39
